In [1]:
import ee
import geemap
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report

PROJECT_ROOT = Path.cwd().parent

ee.Authenticate()
ee.Initialize(project="nigeria-flood-prediction")

In [2]:
# --- Nigeria boundary ---
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
nigeria = countries.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))

# --- Feature 1: Rainfall ---
rainfall = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
rainfall_filtered = rainfall.filterDate("2016-01-01", "2025-12-31")
rainfall_total = rainfall_filtered.sum()
rainfall_nigeria = rainfall_total.clip(nigeria)

# --- Feature 2: Elevation ---
elevation = ee.Image("USGS/SRTMGL1_003")
elevation_nigeria = elevation.clip(nigeria)

# --- Feature 3: Land cover ---
landcover = ee.Image("ESA/WorldCover/v100/2020")
landcover_nigeria = landcover.clip(nigeria)

# --- Feature 4: Distance to water ---
surface_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
water_occurrence = surface_water.select("occurrence")
water_mask = water_occurrence.gt(50)
distance_to_water = water_mask.fastDistanceTransform().sqrt()
distance_to_water_meters = distance_to_water.clip(nigeria).multiply(30)

# --- Feature 5: Forest loss (2015-2023) ---
forest = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
loss_year = forest.select("lossyear")
forest_loss_recent = loss_year.gte(15)
forest_loss_nigeria = forest_loss_recent.clip(nigeria)

# --- Target: Flood label ---
flood_events = ee.ImageCollection("GLOBAL_FLOOD_DB/MODIS_EVENTS/V1")
flood_events_nigeria = flood_events.filter(ee.Filter.bounds(nigeria))
flood_band = flood_events_nigeria.select("flooded")
flood_count = flood_band.sum()
flood_count_nigeria = flood_count.clip(nigeria)
flood_label = flood_count_nigeria.gt(0)

c:\Users\User\Desktop\FLOOD-RISK-PREDICTOR\.venv\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2023_v1_11! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by UMD/hansen/global_forest_change_2025_v1_13

Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2023_v1_11

  warnings.warn(warning, category=DeprecationWarning)


In [3]:
# --- Combine all features + target into one multi-band image ---
combined = rainfall_nigeria.rename("rainfall") \
    .addBands(elevation_nigeria.rename("elevation")) \
    .addBands(landcover_nigeria.rename("landcover")) \
    .addBands(distance_to_water_meters.rename("distance_to_water")) \
    .addBands(forest_loss_nigeria.rename("forest_loss")) \
    .addBands(flood_label.rename("flooded"))

print(combined.bandNames().getInfo())

# --- Generate sample points across Nigeria ---
sample_points = ee.FeatureCollection.randomPoints(region=nigeria, points=8000,seed=42)

# --- Extract feature values at each point ---
sampled_data = combined.sampleRegions(
    collection=sample_points,
    scale=5000,
    geometries=True
)

# --- Convert to dataframe ---
df_gee = geemap.ee_to_df(sampled_data, remove_geom=False)
print(df_gee.shape)

['rainfall', 'elevation', 'landcover', 'distance_to_water', 'forest_loss', 'flooded']
(6265, 7)


In [4]:
# --- Remove duplicate rows (from coarse sampling resolution) ---
df_gee = df_gee.drop_duplicates(subset=["rainfall", "elevation", "landcover", "distance_to_water", "forest_loss", "flooded"])

# --- Remove permanent-water pixels (data leakage source) ---
df_gee = df_gee[df_gee["landcover"] != 80]

print(df_gee.shape)
print(df_gee["flooded"].value_counts())

# --- Save the cleaned dataset ---
GEE_DATA_PATH = PROJECT_ROOT / "data" / "gee" / "flood_features_nigeria.csv"
df_gee.to_csv(GEE_DATA_PATH, index=False)

(5606, 7)
flooded
0    5462
1     144
Name: count, dtype: int64


In [5]:
X = df_gee.drop(columns=["flooded", "geo"])
y = df_gee["flooded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

print(X_train.shape)
print(X_test.shape)

(4204, 5)
(1402, 5)


In [9]:
THRESHOLD = 0.018

MODEL_PATH = PROJECT_ROOT / "models" / "flood_classifier_nigeria_gb.pkl"

model_package = {
    "model": model,
    "threshold": THRESHOLD,
    "features": list(X.columns)
}

joblib.dump(model_package, MODEL_PATH)

['c:\\Users\\User\\Desktop\\FLOOD-RISK-PREDICTOR\\models\\flood_classifier_nigeria_gb.pkl']